<a href="https://colab.research.google.com/github/vishalsj5/PPO/blob/main/PPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch gymnasium numpy

In [3]:
!pip install gymnasium[box2d]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 36.0 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
import numpy as np
import gymnasium as gym

env = gym.make('CartPole-v1')
obs = env.observation_space.shape[0]
n = env.action_space.n

print(n)

2


In [5]:
class PolicyNetwork(nn.Module):
  def __init__(self, obsDim, nActions):
    super().__init__()
    hiddenSize=128
    self.ffn=nn.Sequential(
        nn.Linear(obsDim, hiddenSize),
        nn.ReLU(),
        nn.Linear(hiddenSize, nActions)
    )
  def forward(self, obs):
    logits=self.ffn(obs)
    return Categorical(logits=logits)


In [6]:
env=gym.make('CartPole-v1', render_mode="rgb_array")

In [7]:
def computeRewards(rewards):
  #[1,2,3,4]
  arr=rewards.copy()
  sum=0
  for i in range(len(rewards)-1, -1, -1):
    arr[i]=rewards[i]+sum
    sum=arr[i]

  return(arr)

In [8]:
print(computeRewards([1,2,3,4]))

[10, 9, 7, 4]


In [9]:
def collectEpisode(env, policy):
  obsList=[]
  actionList=[]
  logProbList=[]
  rewardList=[]
  booly=False
  obs, _=env.reset()
  while(not booly):
    obs=torch.tensor(obs, dtype=torch.float32)
    dist=policy(obs)
    action=dist.sample()
    logProb=dist.log_prob(action)
    obsList.append(obs)
    actionList.append(action)
    logProbList.append(logProb)
    obs, reward, terminated, truncated, info = env.step(action.item())
    booly=terminated or truncated
    rewardList.append(reward)
  return obsList, actionList, logProbList, rewardList

In [14]:
policy = PolicyNetwork(obs, n)
optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
epsilon = 0.2
numE = 4

for i in range(500):
    obsList, actionList, OlogProbList, rewardList = collectEpisode(env, policy)

    obsTensor=torch.stack(obsList).detach()
    returns=computeRewards(rewardList)
    OlogProbTensor=torch.stack(OlogProbList).detach()
    returnsTensor=torch.tensor(returns, dtype=torch.float32)
    actionTensor=torch.stack(actionList).detach()

    for j in range(numE):

      dist2=policy(obsTensor)
      lPlist=dist2.log_prob(actionTensor)
      r=torch.exp(lPlist-OlogProbTensor)
      eps=.2
      cr=torch.clamp(r, 1-eps, 1+eps)

      loss = -torch.min(r * returnsTensor, cr * returnsTensor).sum()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    if (i%25==0):
        print(f"Iteration {i}, episode length: {len(rewardList)}")

Iteration 0, episode length: 24
Iteration 25, episode length: 50
Iteration 50, episode length: 27
Iteration 75, episode length: 131
Iteration 100, episode length: 78
Iteration 125, episode length: 86
Iteration 150, episode length: 48
Iteration 175, episode length: 147
Iteration 200, episode length: 158
Iteration 225, episode length: 229
Iteration 250, episode length: 132
Iteration 275, episode length: 46
Iteration 300, episode length: 386
Iteration 325, episode length: 177
Iteration 350, episode length: 298
Iteration 375, episode length: 335
Iteration 400, episode length: 324
Iteration 425, episode length: 237
Iteration 450, episode length: 357
Iteration 475, episode length: 475
